# Train NGAV Anomaly Model
Train an Isolation Forest model from normal process snapshots to score suspicious process behaviors on Windows and Linux endpoints.

In [24]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [17]:
data_path = Path('normal_processes.csv')
df = pd.read_csv(data_path)
print(f'Loaded {len(df)} rows from {data_path.resolve()}')
df.head()

Loaded 309 rows from /home/phuong/btl_ngav/notebooks/normal_processes.csv


,pid,ppid,name,exe,username,status,create_time,cpu_percent,memory_rss,num_threads,num_fds,cmdline_len,is_system_path,is_temp_path,platform,exe_sha256
0,1,0,systemd,/usr/lib/systemd/systemd,root,sleeping,1.780307e+09,0.0,15929344,1,-1,73,1,0,linux,594f5de1a2b5eeb3650bc7a77a7e3137abed3a621c7744...
1,2,0,kthreadd,NaN,root,sleeping,1.780307e+09,0.0,0,1,-1,0,0,0,linux,NaN
2,3,2,pool_workqueue_release,NaN,root,sleeping,1.780307e+09,0.0,0,1,-1,0,0,0,linux,NaN
3,4,2,kworker/R-rcu_gp,NaN,root,idle,1.780307e+09,0.0,0,1,-1,0,0,0,linux,NaN
4,5,2,kworker/R-sync_wq,NaN,root,idle,1.780307e+09,0.0,0,1,-1,0,0,0,linux,NaN


## Data Quality Check
Check nulls and types before building features.

In [18]:
print(df.info())
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0].head(20)

<class 'pandas.DataFrame'>
RangeIndex: 309 entries, 0 to 308
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   pid             309 non-null    int64  
 1   ppid            309 non-null    int64  
 2   name            309 non-null    str    
 3   exe             131 non-null    str    
 4   username        309 non-null    str    
 5   status          309 non-null    str    
 6   create_time     309 non-null    float64
 7   cpu_percent     309 non-null    float64
 8   memory_rss      309 non-null    int64  
 9   num_threads     309 non-null    int64  
 10  num_fds         309 non-null    int64  
 11  cmdline_len     309 non-null    int64  
 12  is_system_path  309 non-null    int64  
 13  is_temp_path    309 non-null    int64  
 14  platform        309 non-null    str    
 15  exe_sha256      131 non-null    str    
dtypes: float64(2), int64(8), str(6)
memory usage: 38.8 KB
None


exe           178
exe_sha256    178
dtype: int64

In [19]:
feature_candidates = [
    'name', 'exe', 'username', 'status', 'platform',
    'pid', 'ppid', 'create_time', 'cpu_percent', 'memory_rss',
    'num_threads', 'num_fds', 'cmdline_len', 'is_system_path', 'is_temp_path'
]

present_features = [f for f in feature_candidates if f in df.columns]
X = df[present_features].copy()

categorical_features = [f for f in ['name', 'exe', 'username', 'status', 'platform'] if f in X.columns]
numerical_features = [f for f in X.columns if f not in categorical_features]

print('Categorical:', categorical_features)
print('Numerical:', numerical_features)
X.head()

Categorical: ['name', 'exe', 'username', 'status', 'platform']
Numerical: ['pid', 'ppid', 'create_time', 'cpu_percent', 'memory_rss', 'num_threads', 'num_fds', 'cmdline_len', 'is_system_path', 'is_temp_path']


,name,exe,username,status,platform,pid,ppid,create_time,cpu_percent,memory_rss,num_threads,num_fds,cmdline_len,is_system_path,is_temp_path
0,systemd,/usr/lib/systemd/systemd,root,sleeping,linux,1,0,1.780307e+09,0.0,15929344,1,-1,73,1,0
1,kthreadd,NaN,root,sleeping,linux,2,0,1.780307e+09,0.0,0,1,-1,0,0,0
2,pool_workqueue_release,NaN,root,sleeping,linux,3,2,1.780307e+09,0.0,0,1,-1,0,0,0
3,kworker/R-rcu_gp,NaN,root,idle,linux,4,2,1.780307e+09,0.0,0,1,-1,0,0,0
4,kworker/R-sync_wq,NaN,root,idle,linux,5,2,1.780307e+09,0.0,0,1,-1,0,0,0


In [20]:
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)
print(f'Train rows: {len(X_train)}, Test rows: {len(X_test)}')

Train rows: 247, Test rows: 62


In [21]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore'))
            ]),
            categorical_features
        ),
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]),
            numerical_features
        )
    ],
    remainder='drop'
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', IsolationForest(n_estimators=300, contamination='auto', random_state=42))
])

pipeline.fit(X_train)
print('Model training completed.')

Model training completed.


In [22]:
train_scores = -pipeline.decision_function(X_train)
threshold = float(np.quantile(train_scores, 0.98))

test_scores = -pipeline.decision_function(X_test)
test_pred = (test_scores > threshold).astype(int)

print(f'Anomaly threshold: {threshold:.6f}')
print(f'Flagged in test set: {test_pred.mean() * 100:.2f}%')
pd.DataFrame({'score': test_scores}).describe()

Anomaly threshold: -0.175419
Flagged in test set: 0.00%


,score
count,62.000000
mean,-0.199909
std,0.006333
min,-0.205313
25%,-0.205313
50%,-0.202600
75%,-0.197855
max,-0.180315


In [23]:
bundle = {
    'pipeline': pipeline,
    'threshold': threshold,
    'feature_columns': present_features,
    'categorical_features': categorical_features,
    'numerical_features': numerical_features
}

output_path = Path('../models/ngav.pkl')
output_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(bundle, output_path)
print(f'Saved model bundle to {output_path.resolve()}')

Saved model bundle to /home/phuong/btl_ngav/models/ngav.pkl


### Data Analysis Key Findings
- The model is trained from normal endpoint process behavior and uses anomaly scoring instead of requiring labeled malware samples.
- Feature preprocessing is fit on train split first, then applied to test split, preventing data leakage.
- The saved bundle includes pipeline, threshold, and feature schema for direct use in the runtime agent.

### Insights or Next Steps
- Collect baseline data from more endpoints and different time windows to reduce false positives.
- Add analyst feedback labels (true/false positive) to move from pure anomaly detection to hybrid supervised detection.